# Data Preparation & Corpus Building

Create stratified sample, build retrieval corpus, and initialize utility functions for downstream retrieval systems (BM25 and semantic search).

**Sections:**
1. Create stratified sample
2. Build corpus
3. Create utilities

## 1. Create Stratified Sample

Filter and create balanced 20,000 review sample across rating distributions. Output: `data/processed/books_sample.parquet`

In [1]:
import pandas as pd
from pathlib import Path
import gc
import sys

# Setup
notebook_dir = Path.cwd()
project_root = notebook_dir.parent if 'notebooks' in str(notebook_dir) else notebook_dir
data_raw = project_root / "data" / "raw"
data_processed = project_root / "data" / "processed"
data_processed.mkdir(parents=True, exist_ok=True)

CATEGORY = "Books"
REVIEWS_FILE = data_raw / f"{CATEGORY}.jsonl.gz"
OUTPUT_FILE = data_processed / f"{CATEGORY.lower()}_sample.parquet"

print(f"{'='*70}")
print(f"Loading {CATEGORY} reviews on MacBook M4 (M1-optimized)")
print(f"{'='*70}")
print(f"Source: {REVIEWS_FILE}")
print(f"Target: {OUTPUT_FILE}")
print()

# Monitor memory
def print_memory():
    import psutil
    process = psutil.Process()
    memory_mb = process.memory_info().rss / 1024 / 1024
    print(f"  Memory usage: {memory_mb:.0f} MB")

# Chunked reading
chunk_size = 5000
chunks = []
total_rows = 0

print(f"Reading in chunks of {chunk_size:,} rows...")
for chunk_num, chunk in enumerate(pd.read_json(REVIEWS_FILE, 
                                               lines=True, 
                                               compression='gzip',
                                               chunksize=chunk_size)):
    total_rows += len(chunk)
    
    # Filter: valid text, minimum length
    chunk = chunk[chunk['text'].notna()]
    chunk = chunk[chunk['text'].astype(str).str.len() >= 20]
    
    chunks.append(chunk)
    
    # Progress
    total_kept = sum(len(c) for c in chunks)
    print(f"   Chunk {chunk_num + 1}: processed {total_rows:>8,} | kept {total_kept:>8,}", end="")
    print_memory()
    
    # Cleanup
    del chunk
    gc.collect()
    
    # Stop at 20k
    if total_kept >= 20000:
        print(f"  Reached target!")
        break

# Combine
print(f"\nCombining {len(chunks)} chunks...", end="")
df = pd.concat(chunks, ignore_index=True)
print(f" Done! {len(df):,} rows")

# Limit to 20k
if len(df) > 20000:
    df = df.head(20000)

# Save
print(f"\nSaving to Parquet (zstd compression)...", end="")
df.to_parquet(OUTPUT_FILE, compression='zstd')
print(f" Done!")

# Summary
file_size_mb = OUTPUT_FILE.stat().st_size / 1024 / 1024
print(f"\n{'='*70}")
print(f"SUCCESS!")
print(f"{'='*70}")
print(f"Rows saved: {len(df):,}")
print(f"File size: {file_size_mb:.2f} MB")
print(f"Columns: {', '.join(df.columns[:5])}...")
print()

Loading Books reviews on MacBook M4 (M1-optimized)
Source: /Users/esteki/Desktop/MDS/575/DSCI_575_milestone1_jchuang_esteki/data/raw/Books.jsonl.gz
Target: /Users/esteki/Desktop/MDS/575/DSCI_575_milestone1_jchuang_esteki/data/processed/books_sample.parquet

Reading in chunks of 5,000 rows...
   Chunk 1: processed    5,000 | kept    4,741  Memory usage: 211 MB
   Chunk 2: processed   10,000 | kept    9,681  Memory usage: 255 MB
   Chunk 3: processed   15,000 | kept   14,284  Memory usage: 265 MB
   Chunk 4: processed   20,000 | kept   18,879  Memory usage: 274 MB
   Chunk 5: processed   25,000 | kept   23,596  Memory usage: 283 MB
  Reached target!

Combining 5 chunks... Done! 23,596 rows

Saving to Parquet (zstd compression)... Done!

SUCCESS!
Rows saved: 20,000
File size: 7.80 MB
Columns: rating, title, text, images, asin...



## 2. Build Corpus

Combine product title and review text into unified documents for retrieval. Output: `data/processed/corpus.pkl`

In [2]:
import pickle
from pathlib import Path
import pandas as pd

# Get project root
notebook_dir = Path.cwd()
if 'notebooks' in str(notebook_dir):
    project_root = notebook_dir.parent
else:
    project_root = notebook_dir

data_processed = project_root / "data" / "processed"

# Load sample
df = pd.read_parquet(data_processed / "books_sample.parquet")

corpus = []
for idx, row in df.iterrows():
    title = str(row.get('title', '')) if pd.notna(row.get('title')) else ""
    text = str(row.get('text', '')) if pd.notna(row.get('text')) else ""
    doc = title + " " + text
    corpus.append(doc)

print(f"Corpus built: {len(corpus)} documents")

with open(data_processed / "corpus.pkl", 'wb') as f:
    pickle.dump(corpus, f)

print(f"Corpus saved to {data_processed / 'corpus.pkl'}")

Corpus built: 20000 documents
Corpus saved to /Users/esteki/Desktop/MDS/575/DSCI_575_milestone1_jchuang_esteki/data/processed/corpus.pkl


## 3. Create src/utils.py

Create utility functions for data loading, text preprocessing, and tokenization. Output: `src/utils.py`

In [3]:
from pathlib import Path

# Get project root
notebook_dir = Path.cwd()
if 'notebooks' in str(notebook_dir):
    project_root = notebook_dir.parent
else:
    project_root = notebook_dir

src_dir = project_root / "src"
src_dir.mkdir(parents=True, exist_ok=True)

utils_code = '''import pandas as pd
import pickle
import string
from typing import List

def load_data(parquet_path: str) -> pd.DataFrame:
    """Load prepared data from parquet file."""
    return pd.read_parquet(parquet_path)

def load_corpus(pkl_path: str) -> List[str]:
    """Load corpus from pickle file."""
    with open(pkl_path, 'rb') as f:
        return pickle.load(f)

def save_corpus(corpus: List[str], pkl_path: str):
    """Save corpus to pickle file."""
    with open(pkl_path, 'wb') as f:
        pickle.dump(corpus, f)

def preprocess_text(text: str) -> str:
    """Apply text preprocessing: lowercase, remove punctuation."""
    text = text.lower()
    translator = str.maketrans('', '', string.punctuation)
    text = text.translate(translator)
    return ' '.join(text.split())

def tokenize(text: str) -> List[str]:
    """Tokenize text by whitespace. Used for BM25."""
    return preprocess_text(text).split()
'''

with open(src_dir / "utils.py", 'w') as f:
    f.write(utils_code)

print(f"Created {src_dir / 'utils.py'}")

Created /Users/esteki/Desktop/MDS/575/DSCI_575_milestone1_jchuang_esteki/src/utils.py
